# Example-03: FFT frequency estimation & zero padding

In [1]:
# Import

import numpy
import torch
import nufft
import yaml

import sys
sys.path.append('..')

from harmonica.util import LENGTH
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# FFT frequency estimation is based on location of max bin in the amplitude spectum

# Set parameters (64 signals with length 1024)

size, length = 64, 1024

# Set window

w = Window.from_cosine(length, order=1.0, dtype=dtype, device=device)
print(w)

# Set TbT data (64 signals with two components and different amplitudes)

t = torch.linspace(1.0, length, length, dtype=dtype, device=device)
data = torch.stack([i*torch.sin(2.0*numpy.pi*1*0.12*t) + 0.01*i*torch.sin(2.0*numpy.pi*2*0.12*t) for i in range(1, size + 1)])
d = Data.from_data(w, data)
print(d)

# Initialize Frequency instances with and without padding

f1 = Frequency(d, pad=length*2**0)
f2 = Frequency(d, pad=length*2**4)
print(f1)
print(f2)

# Apply window (note, window is applied to work)

d.window_remove_mean()
d.window_apply()

# Estimate frequency

f1('fft')
f2('fft')

# Compare results

print(torch.abs(f1.frequency.mean() - 0.12))
print(torch.abs(f2.frequency.mean() - 0.12))

# In this case call invokes task_fft method

f1.compute_fft()
f2.compute_fft()

# Compare results

print(torch.abs(f1.fft_frequency.mean() - 0.12))
print(torch.abs(f2.fft_frequency.mean() - 0.12))

# Clean

del w
del t, data
del d
del f1, f2
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(1024, 'cosine_window', 1.0)
Data(64, Window(1024, 'cosine_window', 1.0))
Frequency(Data(64, Window(1024, 'cosine_window', 1.0)), f_range=(0.0, 0.5))
Frequency(Data(64, Window(1024, 'cosine_window', 1.0)), f_range=(0.0, 0.5))
tensor(1.171875000000e-04, dtype=torch.float64)
tensor(4.882812499996e-06, dtype=torch.float64)
tensor(1.171875000000e-04, dtype=torch.float64)
tensor(4.882812499996e-06, dtype=torch.float64)
